In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy import stats




In [2]:
ANN_FACTOR = 252
BOOTSTRAP_SAMPLES = 1000

In [3]:



# ----------------------------
# Load Equity Curve
# ----------------------------

def load_equity_curve(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.sort_values("Date")
    df["Equity"] = pd.to_numeric(df["Equity"], errors="coerce")
    df["Return"] = df["Equity"].pct_change()
    return df


# ----------------------------
# Build Regimes Internally
# ----------------------------

def build_regimes(start_date, end_date):

    data = yf.download(
        ["^VIX", "HYG", "LQD"],
        start=start_date,
        end=end_date,
        auto_adjust=False, progress=False
    )["Adj Close"]

    data = data.dropna()

    data["credit_spread"] = data["HYG"] - data["LQD"]

    # Rolling 1-year percentile
    data["vix_pct"] = data["^VIX"].rolling(252).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1]
    )

    data["spread_pct"] = data["credit_spread"].rolling(252).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1]
    )

    vol_high = data["vix_pct"] > 0.7
    spread_wide = data["spread_pct"] > 0.7

    data["Regime"] = "Stable Risk-On"
    data.loc[(~vol_high) & spread_wide, "Regime"] = "Fragile"
    data.loc[(vol_high) & (~spread_wide), "Regime"] = "Vol Shock"
    data.loc[(vol_high) & (spread_wide), "Regime"] = "Crisis"

    regimes = data[["Regime"]].reset_index()
    regimes.columns = ["Date", "Regime"]

    return regimes


# ----------------------------
# Metrics
# ----------------------------

def compute_metrics(df):
    df = df.dropna(subset=["Return"])
    if len(df) < 30:
        return None

    total_return = df["Equity"].iloc[-1] / df["Equity"].iloc[0] - 1

    cagr = (df["Equity"].iloc[-1] / df["Equity"].iloc[0]) ** (
        ANN_FACTOR / len(df)
    ) - 1

    r = df["Return"].values
    r_std = np.std(r)

    sharpe = (np.mean(r) / r_std) * np.sqrt(ANN_FACTOR) if r_std > 0 else np.nan

    roll_max = df["Equity"].cummax()
    max_dd = (df["Equity"] / roll_max - 1).min()

    return {
        "TotalReturn": total_return,
        "CAGR": cagr,
        "Sharpe": sharpe,
        "MaxDD": max_dd,
        "MeanDailyReturn": np.mean(r),
        "StdDailyReturn": r_std,
        "Days": len(df),
    }


# ----------------------------
# Statistical Tests
# ----------------------------

def ttest_vs_full(regime_returns, full_returns):
    _, p_val = stats.ttest_ind(
        regime_returns,
        full_returns,
        equal_var=False,
        nan_policy="omit"
    )
    return p_val


def bootstrap_sharpe_ci(returns):
    sharpes = []

    for _ in range(BOOTSTRAP_SAMPLES):
        sample = np.random.choice(returns, size=len(returns), replace=True)
        if np.std(sample) > 0:
            s = (np.mean(sample) / np.std(sample)) * np.sqrt(ANN_FACTOR)
            sharpes.append(s)

    if not sharpes:
        return np.nan, np.nan

    return np.percentile(sharpes, 5), np.percentile(sharpes, 95)

def allocation_decision_economic(row, baseline_sharpe):

    sharpe = row["Sharpe"]
    days = row["Days"]
    maxdd = row["MaxDD"]

    if days < 150:
        return "Insufficient Data"

    delta = sharpe - baseline_sharpe

    if sharpe < 0:
        return "Avoid"

    if delta > 0.25:
        return "Increase Size"

    if delta > 0:
        return "Mild Increase"

    if abs(delta) <= 0.25:
        return "Neutral"

    return "Reduce Size"

# ----------------------------
# Main Runner
# ----------------------------

def run_regime_report(equity_csv: str) -> pd.DataFrame:
    eq = load_equity_curve(equity_csv)

    regimes = build_regimes(
        start_date=eq["Date"].min(),
        end_date=eq["Date"].max(),
    )

    df = eq.merge(regimes, on="Date", how="inner").dropna(subset=["Equity", "Return", "Regime"])

    results = {}

    # Baseline
    full_metrics = compute_metrics(df)
    if full_metrics is None:
        raise RuntimeError("Not enough merged data to compute baseline metrics.")
    results["No Gate (Full Period)"] = full_metrics

    full_returns = df["Return"].dropna()

    # Per-regime
    for regime_name in sorted(df["Regime"].unique()):
        sub = df[df["Regime"] == regime_name].copy()
        m = compute_metrics(sub)
        if m is None:
            continue

        r = sub["Return"].dropna()
        m["MeanReturn_pValue"] = ttest_vs_full(r, full_returns)

        ci_low, ci_high = bootstrap_sharpe_ci(r)
        m["Sharpe_CI_Low"] = ci_low
        m["Sharpe_CI_High"] = ci_high

        results[regime_name] = m

    out = pd.DataFrame(results).T

    # Allocation labels
    full_sharpe = float(out.loc["No Gate (Full Period)", "Sharpe"])
    out["AllocationSignal"] = [
        "Baseline" if idx == "No Gate (Full Period)" else allocation_decision_economic(row, full_sharpe)
        for idx, row in out.iterrows()
    ]

    # Helpful ordering
    preferred = ["No Gate (Full Period)", "Stable Risk-On", "Fragile", "Vol Shock", "Crisis"]
    ordered = [x for x in preferred if x in out.index] + [x for x in out.index if x not in preferred]
    out = out.loc[ordered]

    return out




In [4]:


report = run_regime_report(
    #"data/carry.csv"
    #"data/trend-commodity-vt.csv",
    #"data/trend-reit-vt.csv",
    #"data/trend-long-short-futures-oil.csv",   
    #"data/carry.csv"
    #"data/trend-equity-vt.csv"
    #"data/pairs_portfolio_equity_curve.csv"
    "data/volatility-triple-harvester.csv"
)


In [10]:

def show_regime_report(report: pd.DataFrame):
    """
    Pretty Jupyter display for the DataFrame returned by run_regime_report(...).

    Expects (some subset of) columns:
      TotalReturn, CAGR, Sharpe, MaxDD, Days, AllocationSignal
      MeanReturn_pValue (optional)
      Sharpe_CI_Low, Sharpe_CI_High (optional)
    """
    df = report.copy()

    # ---- Human-friendly percent columns
    if "TotalReturn" in df.columns:
        df["Total Return (%)"] = df["TotalReturn"] * 100
    if "CAGR" in df.columns:
        df["CAGR (%)"] = df["CAGR"] * 100
    if "MaxDD" in df.columns:
        df["Max DD (%)"] = df["MaxDD"] * 100

    # ---- Build display columns (only include what exists)
    colmap = [
        ("AllocationSignal", "Action"),
        ("Days", "Days"),
        ("Total Return (%)", "Total Return (%)"),
        ("CAGR (%)", "CAGR (%)"),
        ("Sharpe", "Sharpe"),
        ("Max DD (%)", "Max DD (%)"),
        ("MeanReturn_pValue", "p-value (mean vs full)"),
        ("Sharpe_CI_Low", "Sharpe CI (5%)"),
        ("Sharpe_CI_High", "Sharpe CI (95%)"),
    ]
    cols = [c for c, _ in colmap if c in df.columns]
    df = df[cols].rename(columns=dict((c, new) for c, new in colmap if c in cols))

    # ---- Formatting rules
    fmt = {}
    for c in ["Total Return (%)", "CAGR (%)", "Max DD (%)"]:
        if c in df.columns:
            fmt[c] = "{:,.1f}"
    if "Sharpe" in df.columns:
        fmt["Sharpe"] = "{:,.2f}"
    if "p-value (mean vs full)" in df.columns:
        fmt["p-value (mean vs full)"] = "{:.3f}"
    for c in ["Sharpe CI (5%)", "Sharpe CI (95%)"]:
        if c in df.columns:
            fmt[c] = "{:,.2f}"

    def style_action(val):
        # light, readable highlights
        if val == "Increase Size":
            return "background-color:#e6ffed; font-weight:700;"
        if val == "Neutral":
            return "background-color:#fff7e6; font-weight:700;"
        if val == "Reduce Size":
            return "background-color:#fff0f0; font-weight:700;"
        if val == "Avoid":
            return "background-color:#ffe6e6; font-weight:700;"
        if val == "Insufficient Data":
            return "background-color:#f2f2f2;"
        if val == "Baseline":
            return "background-color:#e8f1ff; font-weight:700;"
        return ""

    def style_p(val):
        try:
            v = float(val)
        except Exception:
            return ""
        if np.isnan(v):
            return ""
        if v < 0.01:
            return "font-weight:700; color:#0b5;"
        if v < 0.05:
            return "font-weight:700; color:#1a73e8;"
        if v < 0.10:
            return "font-weight:700; color:#b26a00;"
        return "color:#666;"

    # ---- Build styler
    styler = (
        df.style
          .format(fmt)
          .set_table_styles([
              {"selector":"th", "props":[("text-align","left"), ("font-weight","700")]},
              {"selector":"td", "props":[("text-align","right")]},
              {"selector":"caption", "props":[("caption-side","top"), ("font-size","16px"), ("font-weight","700")]},
          ])
          .set_properties(subset=["Action"] if "Action" in df.columns else [], **{"text-align":"left"})
          .set_caption("Regime Performance Report (Baseline + 4 Regimes)")
    )

    if "Action" in df.columns:
        styler = styler.applymap(style_action, subset=["Action"])
    if "p-value (mean vs full)" in df.columns:
        styler = styler.applymap(style_p, subset=["p-value (mean vs full)"])

    # Optional: visual bars for intuition
    for bar_col in ["CAGR (%)", "Total Return (%)", "Sharpe"]:
        if bar_col in df.columns:
            styler = styler.bar(subset=[bar_col], align="mid")

    return styler

# Usage:
# report = run_regime_report("strategy_equity.csv")
show_regime_report(report)


/var/folders/nf/0f78fmr95hz6c9cwnjjzzdlr0000gp/T/ipykernel_31762/3552588550.py:93: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = styler.applymap(style_action, subset=["Action"])
/var/folders/nf/0f78fmr95hz6c9cwnjjzzdlr0000gp/T/ipykernel_31762/3552588550.py:95: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = styler.applymap(style_p, subset=["p-value (mean vs full)"])


,Action,Days,Total Return (%),CAGR (%),Sharpe,Max DD (%),p-value (mean vs full),Sharpe CI (5%),Sharpe CI (95%)
No Gate (Full Period),Baseline,946.000000,38.8,9.1,16.64,-0.3,nan,nan,nan
Stable Risk-On,Reduce Size,557.000000,38.9,16.0,14.82,-0.2,0.164,13.16,16.33
Fragile,Increase Size,204.000000,24.3,30.9,21.76,-0.0,0.086,20.01,24.01
Vol Shock,Insufficient Data,108.000000,18.6,49.0,16.97,-0.0,0.366,13.90,21.24
Crisis,Insufficient Data,77.000000,17.8,70.9,20.18,0.0,0.345,17.63,23.65


In [11]:
def add_regime_ranking(report: pd.DataFrame):

    df = report.copy()

    # Extract baseline Sharpe
    baseline_sharpe = df.loc["No Gate (Full Period)", "Sharpe"]

    # Only rank actual regimes
    regimes = df.drop(index="No Gate (Full Period)").copy()

    # Sharpe delta
    regimes["Sharpe Delta"] = regimes["Sharpe"] - baseline_sharpe

    # Primary ranking by Sharpe (descending)
    regimes["Regime Rank"] = (
        regimes["Sharpe"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    # Sort by rank
    regimes = regimes.sort_values("Regime Rank")

    # Add baseline back
    df.update(regimes)

    # Combine clean output
    final = pd.concat([
        df.loc[["No Gate (Full Period)"]],
        regimes
    ])

    return final

In [12]:
def rank_to_multiplier(rank, total_regimes):
    return 1 + (total_regimes - rank) * 0.1

In [13]:
ranked = add_regime_ranking(report)

In [14]:
regimes_only = ranked.drop(index="No Gate (Full Period)")

total_regimes = len(regimes_only)

regimes_only["Multiplier"] = regimes_only["Regime Rank"].apply(
    lambda r: rank_to_multiplier(r, total_regimes)
)

regimes_only

,TotalReturn,CAGR,Sharpe,MaxDD,MeanDailyReturn,StdDailyReturn,Days,MeanReturn_pValue,Sharpe_CI_Low,Sharpe_CI_High,AllocationSignal,Sharpe Delta,Regime Rank,Multiplier
Fragile,0.243429,0.308834,21.760530,-0.000199,0.000385,0.000281,204.0,0.085507,20.014392,24.006275,Increase Size,5.122194,1.0,1.3
Crisis,0.177971,0.709244,20.179061,0.000000,0.000381,0.000300,77.0,0.345202,17.625784,23.651371,Insufficient Data,3.540725,2.0,1.2
Vol Shock,0.186307,0.489796,16.968363,-0.000446,0.000379,0.000355,108.0,0.366044,13.897188,21.240445,Insufficient Data,0.330027,3.0,1.1
Stable Risk-On,0.388551,0.160108,14.824421,-0.002263,0.000321,0.000344,557.0,0.164159,13.157358,16.328344,Reduce Size,-1.813915,4.0,1.0
